# Lecture 10 — Filters & Interactivity
## When Filters Help and When They Hurt

> **Dataset:** CO₂ Emissions 2000-2022

> **Big Book of Dashboards:** Ch.8 Multiple KPIs

---
## Opening: Model Answer Review

## Steps to Deploy on Streamlit Community Cloud:

1. **Create a `requirements.txt`** in the repo root listing your dependencies

2. **Fix the data path before pushing** — `../data/` works locally but breaks on Community Cloud. Replace it with a `pathlib` path that resolves relative to the script file itself:
    ```python
    from pathlib import Path
    df = pd.read_csv(Path(__file__).parent.parent / 'data' / 'world_happiness_2023.csv')
    ```
    This works identically locally and on Community Cloud — `Path(__file__).parent.parent` always points to the repo root regardless of where the script is launched from

3. **Push your app to GitHub** — your `.py` file and `requirements.txt` must both be committed and pushed. The `data/` folder must also be in the repo


4. **Go to [share.streamlit.io](https://share.streamlit.io)** and sign in with your GitHub account


5. **Click "New app"** and fill in:
    - **Repository:** `your-github-username/your-repo-name`
    - **Branch:** `main`
    - **Main file path:** `week09/dashboard.py`


6. **Click Deploy** — Community Cloud installs your requirements and launches the app. You get a public URL to share

---
## Design Principles - Part I: Interactivity Tips



**💡 It is best practice to have filters grouped in one place — at the top of the page or in a dedicated sidebar. Why?: users should be able to see all available filters at once, set them, and then view the results**

**💡 more interactivity is not always better!!! Design for the most common use case by default --> Power users can adjust; casual users should get the right answer immediately**

**💡 BBD Filter Best Practices:**

| Rule | Why |
|---|---|
| All filters in one place | Users see all options simultaneously |
| Default to the most common case | Most users want 'all regions', not a blank |
| Show how many records match | 'Showing 12 of 15 countries' reduces confusion |
| Handle empty results | `st.warning()` + `st.stop()` — never show an empty chart silently |
| Chained filters | Region narrows country list — reduces cognitive load |

---
## Design Principles - Part II: Widget Types


| Widget | Returns | Use for | Watch out |
|---|---|---|---|
| `st.selectbox` | String | Single choice from a list | Returns the value itself, not the index |
| `st.multiselect` | List | Multiple choices | Returns `[]` when nothing selected — always check before filtering |
| `st.slider(min, max, (lo,hi))` | Tuple | Numeric range filter | Pass a tuple as default for a range slider; single value for a point slider |
| `st.date_input(label, value=(start,end))` | Tuple of dates | Calendar date range | Returns `datetime.date` — convert with `pd.Timestamp` before pandas comparisons |
| `st.radio` | String | 2–4 mutually exclusive options | Avoid for long lists — use `selectbox` instead |
| `st.checkbox` | Boolean | On/off toggle | Default is `False` — make sure your logic handles the unchecked state |

**`st.slider` vs `st.date_input`:** use slider when data is aggregated to integer years;
use `date_input` when your dataset has actual timestamps (daily, hourly data).



---
## Let's Code An Example Dashboard 💻 

> **HOW TO RUN:** Copy each block into `app.py`, run `streamlit run app.py`.


In [ ]:
import streamlit as st
import pandas as pd
import plotly.express as px
from pathlib import Path

# ── Data ──────────────────────────────────────────────────────────────────────
# @st.cache_data: Streamlit reruns the entire script on every widget interaction.
# Without caching, the CSV is read from disk on every interaction — slow and wasteful.
# cache_data stores the result after the first run and reuses it until the file changes

@st.cache_data
def load_data():
    path = Path(__file__).parent.parent / 'data' / 'co2_emissions.csv'
    df = pd.read_csv(path)
    df['Date'] = pd.to_datetime(df['Year'].astype(str) + '-01-01')
    return df

df = load_data()

In [ ]:
# ── STEP 1: multiselect with empty-state guard ────────────────────────────
st.title("CO₂ Emissions Explorer")

with st.sidebar:
    st.header("Filters")
    selected_countries = st.multiselect(
        "Countries",
        options=sorted(df['Country'].unique()),
        default=['China', 'United States', 'India', 'Germany']
    )

if not selected_countries:
    st.warning("👆 Select at least one country.")
    st.stop()   # nothing below runs — avoids confusing empty chart

filtered = df[df['Country'].isin(selected_countries)]
st.caption(f"Showing {len(selected_countries)} countries | {len(filtered)} data points")



In [ ]:
# ── STEP 2: numeric range slider ─────────────────────────────────────────
with st.sidebar:
    st.header("Filters")
    selected_countries = st.multiselect(
        "Countries", sorted(df['Country'].unique()),
        default=['China', 'United States', 'India', 'Germany']
    )
    # Tuple default → two-handle range slider
    year_range = st.slider("Year range",
        int(df['Year'].min()), int(df['Year'].max()), (2000, 2022))

if not selected_countries:
    st.warning("Select at least one country.")
    st.stop()

filtered = (df[df['Country'].isin(selected_countries)]
            .query("@year_range[0] <= Year <= @year_range[1]"))

fig = px.line(filtered, x='Year', y='CO2_Mt', color='Country',
              labels={'CO2_Mt': 'CO2 (Mt)'})
fig.update_layout(plot_bgcolor='white', paper_bgcolor='white',
                  font=dict(family='Arial'))
st.plotly_chart(fig, use_container_width=True)


In [ ]:
# ── STEP 3: st.date_input — calendar picker for real timestamps ───────────
import datetime

st.title("CO₂ Emissions — Date Input Demo")

with st.sidebar:
    st.header("Filters")
    selected_countries = st.multiselect(
        "Countries", sorted(df['Country'].unique()),
        default=['China', 'United States', 'Germany']
    )
    # date_input: use when data has real timestamps (daily/hourly)
    # The CO₂ data has integer years — we converted to dates in the loader
    date_range = st.date_input(
        "Date range",
        value=(datetime.date(2005, 1, 1), datetime.date(2020, 1, 1)),
        min_value=datetime.date(int(df['Year'].min()), 1, 1),
        max_value=datetime.date(int(df['Year'].max()), 1, 1),
        format="YYYY-MM-DD"
    )
    # Guard: user may have clicked start but not end yet
    if len(date_range) != 2:
        st.warning("Select a start AND end date.")
        st.stop()

if not selected_countries:
    st.warning("Select at least one country.")
    st.stop()

# Always convert date → pd.Timestamp before pandas comparisons
start_ts, end_ts = pd.Timestamp(date_range[0]), pd.Timestamp(date_range[1])
filtered = df[
    df['Country'].isin(selected_countries) &
    (df['Date'] >= start_ts) &
    (df['Date'] <= end_ts)
]

if filtered.empty:
    st.warning("No data in this date range for the selected countries.")
    st.stop()

st.caption(f"{date_range[0].strftime('%d %b %Y')} — {date_range[1].strftime('%d %b %Y')}")

fig = px.line(filtered, x='Date', y='CO2_Mt', color='Country',
              labels={'CO2_Mt': 'CO2 Emissions (Mt)', 'Date': ''})
fig.update_layout(plot_bgcolor='white', paper_bgcolor='white',
                  font=dict(family='Arial'))
st.plotly_chart(fig, use_container_width=True)


In [ ]:
# ── STEP 4: radio + chained filters ──────────────────────────────────────
with st.sidebar:
    st.header("Filters")
    # Chained filter: Region narrows Country list
    regions = ['All'] + sorted(df['Region'].unique())
    selected_region = st.selectbox("Region", regions)

    if selected_region == 'All':
        country_options = sorted(df['Country'].unique())
    else:
        country_options = sorted(df[df['Region']==selected_region]['Country'].unique())

    selected_countries = st.multiselect("Countries", country_options, default=country_options[:3])
    year_range = st.slider("Year range", int(df['Year'].min()), int(df['Year'].max()), (2000, 2022))

    st.divider()
    # radio: 2-4 exclusive options — clearer than selectbox
    metric = st.radio("Metric", ["Total CO2 (Mt)", "CO2 per capita"])

if not selected_countries:
    st.warning("Select at least one country.")
    st.stop()

filtered = (df[df['Country'].isin(selected_countries)]
            .query("@year_range[0] <= Year <= @year_range[1]"))

y_col = 'CO2_Mt' if metric == "Total CO2 (Mt)" else 'CO2_per_capita'
y_label = 'CO2 Emissions (Mt)' if y_col == 'CO2_Mt' else 'CO2 per Capita'

st.caption(f"{len(selected_countries)} countries | {selected_region} | {year_range[0]}–{year_range[1]} | {metric}")

col1, col2 = st.columns(2)
with col1:
    fig1 = px.line(filtered, x='Year', y=y_col, color='Country',
                   labels={y_col: y_label},
                   title=f'{metric} over time')
    fig1.update_layout(plot_bgcolor='white', paper_bgcolor='white', font=dict(family='Arial'))
    st.plotly_chart(fig1, use_container_width=True)

with col2:
    latest = filtered[filtered['Year']==filtered['Year'].max()].sort_values(y_col)
    fig2 = px.bar(latest, x=y_col, y='Country', orientation='h',
                  color_discrete_sequence=['#2E75B6'],  # BBD: highlight
                  title=f'Latest year ranking')
    fig2.update_layout(plot_bgcolor='white', paper_bgcolor='white', font=dict(family='Arial'),
                       xaxis=dict(range=[0, latest[y_col].max()*1.15]))
    fig2.update_traces(marker_line_width=0)
    st.plotly_chart(fig2, use_container_width=True)


---
## Transition to Exercise

> *'Build the full CO₂ filter dashboard with all five widget types.*
> *Run with `streamlit run lecture10_exercise.py`. Push to `week10/`.'*

**BBD + Streamlit checklist:**
- All filters in sidebar — none scattered in the main area
- At least 3 widget types including `st.date_input`
- Chained filter (region → country)
- `st.stop()` for: empty selection AND incomplete date range
- Convert `date_input` to `pd.Timestamp` before filtering
- Filter summary caption (BBD: show count of matching records)
- `@st.cache_data` on data loader
